<a href="https://colab.research.google.com/github/abhisheksvcs25-bms/Bio-Inspired-Systems/blob/YT-Hand-Gesture-Controller/Grey_Wolf_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load dataset
file_path = "/content/SYNTHETIC_EV_DATA.csv"

if not os.path.exists(file_path):
    file_path = "/SYNTHETIC_EV_DATA.csv"

if not os.path.exists(file_path):
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]

data = pd.read_csv(file_path)
data.columns = data.columns.str.strip()

print("Dataset loaded successfully!")
print("Dataset shape:", data.shape)
print("Columns:", list(data.columns))

# 2. Identify target column
target_candidates = [
    "charging time/minutes",
    "charging_duration",
    "charging duration",
    "charging_time",
    "charging time",
    "duration",
    "duration_minutes",
    "charging_duration_minutes"
]

target = None

for col in data.columns:
    if col.lower().strip() in target_candidates:
        target = col
        break

if target is None:
    possible = [
        col for col in data.columns
        if ("charg" in col.lower() or "duration" in col.lower())
        and ("time" in col.lower() or "duration" in col.lower())
    ]
    if len(possible) == 1:
        target = possible[0]

if target is None:
    raise ValueError(
        "Target column not identified. Available columns: "
        + str(list(data.columns))
        + ". Set target manually in the code."
    )

print("Target column:", target)

# 3. Clean dataset
data = data.dropna(subset=[target]).copy()
y = pd.to_numeric(data[target], errors="coerce")
valid = y.notna()

data = data.loc[valid].copy()
y = y.loc[valid].values

X = data.drop(columns=[target])

# Remove columns that contain only missing values
X = X.dropna(axis=1, how="all")

# Convert categorical features into numerical values
X = pd.get_dummies(X, drop_first=True, dtype=float)

# Fill missing feature values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)

X = X.astype(float).values

# 4. Split data (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# 5. Scale input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Extreme Learning Machine
def elm_predict(Xtr, ytr, Xnew, neurons, reg):
    rng = np.random.default_rng(42)

    W = rng.normal(0, 1, (Xtr.shape[1], neurons))
    b = rng.normal(0, 1, neurons)

    H = np.tanh(Xtr @ W + b)
    Hnew = np.tanh(Xnew @ W + b)

    beta = np.linalg.solve(
        H.T @ H + reg * np.eye(neurons),
        H.T @ ytr
    )

    return Hnew @ beta

# 7. Fitness function
def fitness(position):
    neurons = int(np.clip(round(position[0]), 5, 50))
    reg = 10 ** np.clip(position[1], -5, 1)

    prediction = elm_predict(
        X_train, y_train, X_train, neurons, reg
    )

    return np.sqrt(mean_squared_error(y_train, prediction))

# 8. Grey Wolf Optimizer
def gwo():
    np.random.seed(42)

    population = 10
    iterations = 20

    lower = np.array([5, -5])
    upper = np.array([50, 1])

    wolves = np.random.uniform(
        lower, upper, (population, 2)
    )

    alpha = beta = delta = None
    alpha_score = beta_score = delta_score = float("inf")

    for t in range(iterations):

        for i in range(population):
            score = fitness(wolves[i])

            if score < alpha_score:
                delta_score, delta = beta_score, beta
                beta_score, beta = alpha_score, alpha
                alpha_score, alpha = score, wolves[i].copy()

            elif score < beta_score:
                delta_score, delta = beta_score, beta
                beta_score, beta = score, wolves[i].copy()

            elif score < delta_score:
                delta_score, delta = score, wolves[i].copy()

        a = 2 - 2 * t / iterations

        for i in range(population):
            positions = []

            for leader in [alpha, beta, delta]:
                A = 2 * a * np.random.rand(2) - a
                C = 2 * np.random.rand(2)
                D = np.abs(C * leader - wolves[i])
                positions.append(leader - A * D)

            wolves[i] = np.mean(positions, axis=0)
            wolves[i] = np.clip(wolves[i], lower, upper)

        print(
            "Iteration", t + 1,
            "Best RMSE:", round(alpha_score, 4)
        )

    return alpha

# 9. Optimize ELM parameters
best_params = gwo()

best_neurons = int(round(best_params[0]))
best_reg = 10 ** best_params[1]

print("\nBest Hidden Neurons:", best_neurons)
print("Best Regularization:", round(best_reg, 6))

# 10. Predict charging duration
y_pred = elm_predict(
    X_train, y_train, X_test,
    best_neurons, best_reg
)

# 11. Evaluate model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n========== GWO-ELM RESULTS ==========")
print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R2 Score:", round(r2, 4))

# 12. Display actual and predicted values
results = pd.DataFrame({
    "Actual Charging Duration": y_test,
    "Predicted Charging Duration": y_pred
})

print("\nPrediction Results:")
print(results.head(10))

# 13. Plot actual vs predicted values
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Charging Duration")
plt.ylabel("Predicted Charging Duration")
plt.title("GWO-ELM EV Charging Duration Prediction")
plt.grid(True)
plt.show()

# 14. Plot prediction comparison
plt.figure(figsize=(9, 5))
plt.plot(y_test[:30], label="Actual")
plt.plot(y_pred[:30], label="Predicted")
plt.xlabel("Test Samples")
plt.ylabel("Charging Duration")
plt.title("Actual vs Predicted Charging Duration")
plt.legend()
plt.grid(True)
plt.show()


Dataset loaded successfully!
Dataset shape: (1965239, 4)
Columns: ['connectionTime_decimal', 'chargingDuration', 'kWhDelivered', 'dayIndicator']
Target column: chargingDuration
Iteration 1 Best RMSE: 2.7954
Iteration 2 Best RMSE: 2.7905
